# Illustrate a simple SQL agent

Using locally loaded LLMs with Ollama

# Setup

<p>In this notebook we will use the following python packages on Python 3.10.15:</p>
<ul>
<li><code>eccore</code></li>
<li><code>jupyter</code></li>
<li><code>matplotlib</code></li>
<li><code>numpy</code></li>
<li><code>ollama</code></li>
<li><code>pandas</code></li>
<li><code>pillow</code></li>
<li><code>scikit-learn</code></li>
</ul>

<details>
    <summary><i>List of Python packages with versions:<i></summary>
<p></br>The notebook was last tested with the following versions:</p>
<ul>
<li><code>eccore==0.0.5</code></li>
<li><code>jupyter==1.1.1</code></li>
<li><code>matplotlib==3.10.7</code></li>
<li><code>numpy==2.3.3</code></li>
<li><code>ollama==0.6.1</code></li>
<li><code>pandas==2.3.3</code></li>
<li><code>pillow==11.3.0</code></li>
<li><code>scikit-learn==1.8.0</code></li>
</ul>
</details>     

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
import re
import sqlite3

from datetime import datetime, timedelta
from IPython.display import display, Markdown, clear_output
from pathlib import Path
from PIL import Image

from eccore.comms import clean_no_proxy_12vpn
clean_no_proxy_12vpn()

'100.64.*,172.16.*,192.168.*,10.*,127.*,169.254.*,localhost,*.local,vpnguistub'

## Setup LLM

In [2]:
import ollama

In [4]:
ollama_client = ollama.Client(host="http://127.0.0.1:11434", trust_env=False)
model_names = sorted([model.model for model in ollama_client.list().models])
model_names

['bge-m3:latest',
 'codegemma:7b',
 'codellama:7b',
 'deepseek-r1:7b',
 'deepseek-r1:8b',
 'gemma3:4b',
 'granite3.2-vision:latest',
 'llama3.1:8b',
 'llama3.2:1b-instruct-fp16',
 'llama3.2:1b-text-fp16',
 'llama3.2:3b',
 'llava:7b',
 'mxbai-embed-large:335m',
 'nomic-embed-text:latest',
 'qwen2.5:3b',
 'qwen2.5:7b',
 'qwen3-embedding:latest',
 'qwen3:8b']

### msg to llm

In [6]:
def send_message_to_llm(message, model, system_message=None, streaming=True, max_tokens=8000, client=ollama_client):
    """Send a message to the LLM and return the response"""
    display(Markdown(f"Model: `{model}`"))
    display(Markdown(f"Answer to: **{message}**"))
    options = {}
    if max_tokens:
        options['num_predict'] = max_tokens
    if streaming:
        output = display(Markdown('Computing ...'), display_id=True)
        stream = client.chat(
            model=model,
            messages=[{'role': 'user', 'content': message}],
            stream=streaming,
            options=options if options else None,
            )
        streamed = ""
        for chunk in stream:
            streamed += chunk['message']['content']
            # clear_output(wait=True)
            output.update(Markdown(streamed))

        print(f"Done! in {len(streamed):,d} characters.")
        return streamed
    else:
        output = display(Markdown('Computing ...'), display_id=True)
        response = client.chat(
            model=model, 
            messages=[{'role': 'user', 'content': message}], 
            stream=False,
            options=options if options else None,
        )
        output.update(Markdown(response['message']['content']))
        print(f"Done! in {len(response['message']['content']):,d} characters.")
        return response

# send_message_to_llm('Capital of France', model='gemma3:4b', streaming=False)
# send_message_to_llm('Capital of France', model='gemma3:4b', streaming=True)

### msg to thinking llm

In [7]:
def send_message_to_llm_with_thinking_stream(message, model, system_message=None, streaming=True, max_tokens=8000,client=ollama_client):
    """Send message and parse thinking vs answer sections while streaming"""
    display(Markdown(f"Model: `{model}`"))
    display(Markdown(f"Answer to: **{message}**"))
    
    options = {}
    if max_tokens:
        options['num_predict'] = max_tokens
    
    if streaming:
        # Create display handles for thinking and answer
        thinking_output = display(Markdown("### 🧠 Thinking Process\n```\n\n```"), display_id=True)
        answer_output = display(Markdown("### 💡 Final Answer\n\n"), display_id=True)
        
        stream = client.chat(
            model=model,
            messages=[{'role': 'user', 'content': message}],
            stream=True,
            options=options if options else None,
        )
        
        streamed = ""
        thinking = ""
        answer = ""
        in_think_tag = False
        
        for chunk in stream:
            content = chunk['message']['content']
            streamed += content
            
            # Check for <think> tag opening
            if '<think>' in streamed and not in_think_tag:
                in_think_tag = True
                # Split at <think>
                parts = streamed.split('<think>', 1)
                if len(parts) > 1:
                    thinking = parts[1]
            
            # Check for </think> tag closing
            elif '</think>' in streamed and in_think_tag:
                in_think_tag = False
                # Split to separate thinking from answer
                parts = streamed.split('</think>', 1)
                if len(parts) > 1:
                    thinking = parts[0].split('<think>', 1)[1] if '<think>' in parts[0] else ""
                    answer = parts[1]
                    # Update both displays
                    thinking_output.update(Markdown(f"### 🧠 Thinking Process\n```\n{thinking}\n```"))
                    answer_output.update(Markdown(f"### 💡 Final Answer\n{answer}"))
            
            # While inside think tags
            elif in_think_tag:
                # Extract thinking content
                if '<think>' in streamed:
                    thinking = streamed.split('<think>', 1)[1]
                    thinking_output.update(Markdown(f"### 🧠 Thinking Process\n```\n{thinking}\n```"))
            
            # After think tags (or no think tags)
            else:
                # Check if we already closed think tags
                if '</think>' in streamed:
                    parts = streamed.split('</think>', 1)
                    if len(parts) > 1:
                        answer = parts[1]
                        answer_output.update(Markdown(f"### 💡 Final Answer\n{answer}"))
                else:
                    # No think tags at all, just show as answer
                    answer_output.update(Markdown(f"### 💡 Final Answer\n{streamed}"))
        
        print(f"Done! in {len(streamed):,d} characters.")
        
        # Final parse to ensure clean separation
        think_match = re.search(r'<think>(.*?)</think>', streamed, re.DOTALL)
        if think_match:
            thinking = think_match.group(1).strip()
            answer = re.sub(r'<think>.*?</think>', '', streamed, flags=re.DOTALL).strip()
        else:
            thinking = None
            answer = streamed
        
        return {'thinking': thinking, 'answer': answer, 'full': streamed}
    
    else:
        # Non-streaming version (unchanged)
        output = display(Markdown('Computing ...'), display_id=True)
        response = client.chat(
            model=model, 
            messages=[{'role': 'user', 'content': message}], 
            stream=False,
            options=options if options else None,
        )
        content = response['message']['content']
        
        think_match = re.search(r'<think>(.*?)</think>', content, re.DOTALL)
        if think_match:
            thinking = think_match.group(1).strip()
            answer = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL).strip()
            
            display(Markdown("### 🧠 Thinking Process"))
            display(Markdown(f"```\n{thinking}\n```"))
            display(Markdown("### 💡 Final Answer"))
            display(Markdown(answer))
        else:
            output.update(Markdown(content))
            thinking = None
            answer = content
        
        print(f"Done! in {len(content):,d} characters.")
        return {'thinking': thinking, 'answer': answer, 'full': content}

# send_message_to_llm_with_thinking_stream('Capital of France', model='gemma3:4b', streaming=False)
# send_message_to_llm_with_thinking_stream('Capital of France', model='gemma3:4b', streaming=True)

### image to VLLM

In [8]:
def send_image_to_llm(message, image_path, model='llava:7b', streaming=True, max_tokens=8000, client=ollama_client):
    """Send a message with an image to a vision LLM"""
    display(Markdown(f"Model: `{model}`"))
    display(Markdown(f"Question: **{message}**"))
    display(Markdown(f"Image File: **{image_path.name}**"))

    im = Image.open(image_path)
    max_size = (512, 512)
    im.thumbnail(max_size, Image.Resampling.LANCZOS)    

    options = {}
    if max_tokens:
        options['num_predict'] = max_tokens
    
    if streaming:
        output = display(Markdown('Computing ...'), display_id=True)
        stream = client.chat(
            model=model,
            messages=[{
                'role': 'user',
                'content': message,
                'images': [image_path]  # Pass image path
            }],
            stream=True,
            options=options if options else None,
        )
        streamed = ""
        for chunk in stream:
            streamed += chunk['message']['content']
            output.update(Markdown(streamed))
        
        print(f"Done! in {len(streamed):,d} characters.")
        display(im)
        return streamed
    else:
        output = display(Markdown('Computing ...'), display_id=True)
        response = client.chat(
            model=model,
            messages=[{
                'role': 'user',
                'content': message,
                'images': [image_path]
            }],
            stream=False,
            options=options if options else None,
        )
        output.update(Markdown(response['message']['content']))
        print(f"Done! in {len(response['message']['content']):,d} characters.")
        display(im)
        return response['message']['content']

# send_image_to_llm('What is in this image?', Path('horse.png'), model='llava:7b', streaming=False)

## Setup sqlite database

### Create a database if it does not exist yet

In [9]:
db_dir = Path('../db')
db_dir.mkdir(exist_ok=True)

# Database path
db_path = db_dir / 'crm.db'

# Remove existing database to start fresh
if db_path.exists():
    db_path.unlink()

# Connect to database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create tables
cursor.execute('''
CREATE TABLE accounts (
    uid INTEGER PRIMARY KEY AUTOINCREMENT,
    account_name TEXT NOT NULL,
    city TEXT NOT NULL,
    country TEXT NOT NULL
)
''')

cursor.execute('''
CREATE TABLE products (
    uid INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL,
    product_price REAL NOT NULL,
    delivery_leadtime INTEGER NOT NULL
)
''')

cursor.execute('''
CREATE TABLE orders (
    uid INTEGER PRIMARY KEY AUTOINCREMENT,
    account_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (account_id) REFERENCES accounts(uid)
)
''')

cursor.execute('''
CREATE TABLE order_details (
    uid INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(uid),
    FOREIGN KEY (product_id) REFERENCES products(uid)
)
''')

# Insert dummy data for accounts
locations = [
    ('Shanghai', 'PRC'),
    ('Beijing', 'PRC'),
    ('Shenzhen', 'PRC'),
    ('Tokyo', 'Japan'),
    ('Seoul', 'Korea')
]

company_prefixes = ['TechStyle', 'FashionHub', 'ModernWear', 'GlobalTextile', 'UrbanThread', 
                   'StyleCraft', 'PremiumFabric', 'ElegantWear', 'TrendSetter', 'ChicOutfit',
                   'ClassicApparel', 'NovaDress', 'EliteGarment', 'SwiftFashion', 'LuxuryFit']

accounts_data = []
for i, prefix in enumerate(company_prefixes):
    city, country = random.choice(locations)
    accounts_data.append((f"{prefix} Inc", city, country))

cursor.executemany('INSERT INTO accounts (account_name, city, country) VALUES (?, ?, ?)', accounts_data)

# Insert dummy data for products (clothing items)
products_data = [
    ('Cotton T-Shirt', 19.99, 3),
    ('Denim Jeans', 59.99, 5),
    ('Leather Jacket', 149.99, 7),
    ('Wool Sweater', 79.99, 4),
    ('Silk Blouse', 89.99, 6),
    ('Linen Pants', 69.99, 4),
    ('Polo Shirt', 39.99, 3),
    ('Dress Shirt', 49.99, 4),
    ('Cargo Shorts', 34.99, 3),
    ('Hooded Sweatshirt', 54.99, 5),
    ('Blazer', 129.99, 6),
    ('Pencil Skirt', 44.99, 4),
    ('Maxi Dress', 99.99, 5),
    ('Cardigan', 64.99, 4),
    ('Chinos', 54.99, 4),
    ('Track Pants', 39.99, 3),
    ('Tank Top', 14.99, 2),
    ('Bomber Jacket', 119.99, 6),
    ('Trench Coat', 179.99, 8),
    ('Cashmere Scarf', 84.99, 5)
]

cursor.executemany('INSERT INTO products (product_name, product_price, delivery_leadtime) VALUES (?, ?, ?)', products_data)

# Insert dummy data for orders
order_statuses = ['Pending', 'Processing', 'Shipped', 'Delivered', 'Cancelled']
base_date = datetime(2024, 1, 1)

orders_data = []
for i in range(50):
    account_id = random.randint(1, len(accounts_data))
    days_offset = random.randint(0, 365)
    order_date = (base_date + timedelta(days=days_offset)).strftime('%Y-%m-%d')
    status = random.choice(order_statuses)
    orders_data.append((account_id, order_date, status))

cursor.executemany('INSERT INTO orders (account_id, order_date, status) VALUES (?, ?, ?)', orders_data)

# Insert dummy data for order_details
order_details_data = []
for order_id in range(1, 51):
    # Each order has 1-5 line items
    num_items = random.randint(1, 5)
    for _ in range(num_items):
        product_id = random.randint(1, len(products_data))
        quantity = random.randint(1, 10)
        order_details_data.append((order_id, product_id, quantity))

cursor.executemany('INSERT INTO order_details (order_id, product_id, quantity) VALUES (?, ?, ?)', order_details_data)

# Commit and close
conn.commit()
conn.close()

print(f"✓ Database created at: {db_path}")
print(f"✓ {len(accounts_data)} accounts inserted")
print(f"✓ {len(products_data)} products inserted")
print(f"✓ {len(orders_data)} orders inserted")
print(f"✓ {len(order_details_data)} order details inserted")

✓ Database created at: ../db/crm.db
✓ 15 accounts inserted
✓ 20 products inserted
✓ 50 orders inserted
✓ 141 order details inserted


### query_db

In [10]:
def query_db(sql_query, db_path='../db/crm.db'):
    """Execute a SQL query and return results as a pandas DataFrame"""
    conn = sqlite3.connect(db_path)
    try:
        df = pd.read_sql_query(sql_query, conn)
        return df
    finally:
        conn.close()


query_db("SELECT * FROM accounts LIMIT 5")

,uid,account_name,city,country
0,1,TechStyle Inc,Tokyo,Japan
1,2,FashionHub Inc,Shenzhen,PRC
2,3,ModernWear Inc,Shenzhen,PRC
3,4,GlobalTextile Inc,Shanghai,PRC
4,5,UrbanThread Inc,Shanghai,PRC


In [11]:
df = query_db(
    "SELECT * FROM sqlite_master WHERE type='table'"
)
db_schema = '\n'.join(df.sql.to_list())
print(db_schema)

CREATE TABLE accounts (
    uid INTEGER PRIMARY KEY AUTOINCREMENT,
    account_name TEXT NOT NULL,
    city TEXT NOT NULL,
    country TEXT NOT NULL
)
CREATE TABLE sqlite_sequence(name,seq)
CREATE TABLE products (
    uid INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL,
    product_price REAL NOT NULL,
    delivery_leadtime INTEGER NOT NULL
)
CREATE TABLE orders (
    uid INTEGER PRIMARY KEY AUTOINCREMENT,
    account_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (account_id) REFERENCES accounts(uid)
)
CREATE TABLE order_details (
    uid INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(uid),
    FOREIGN KEY (product_id) REFERENCES products(uid)
)


### sql_prompt

In [31]:
question = "What are the companies in the database?"

def sql_prompt(question):
    return f""" 
    ### Instructions:
    Your task is to convert a question into a SQL query, given a Postgres database schema.
    Adhere to these rules:
    - **Deliberately go through the question and database schema word by word** to appropriately answer the question
    - **Use Table Aliases** to prevent ambiguity. For example, `SELECT table1.col1, table2.col1 FROM table1 JOIN table2 ON table1.id = table2.id`.
    - When creating a ratio, always cast the numerator as float

    ### Input:
    Generate a SQL query that answers the question `{question}`.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE accounts (
        uid INTEGER PRIMARY KEY AUTOINCREMENT,
        company_name TEXT NOT NULL,
        city TEXT NOT NULL,
        country TEXT NOT NULL
    );

    CREATE TABLE products (
        uid INTEGER PRIMARY KEY AUTOINCREMENT,
        product_name TEXT NOT NULL,
        product_price REAL NOT NULL,
        delivery_leadtime INTEGER NOT NULL
    )
    ;

    CREATE TABLE orders (
        uid INTEGER PRIMARY KEY AUTOINCREMENT,
        account_id INTEGER NOT NULL,
        order_date TEXT NOT NULL,
        status TEXT NOT NULL,
        FOREIGN KEY (account_id) REFERENCES accounts(uid)
    )
    ;

    CREATE TABLE order_details (
        uid INTEGER PRIMARY KEY AUTOINCREMENT,
        order_id INTEGER NOT NULL,
        product_id INTEGER NOT NULL,
        quantity INTEGER NOT NULL,
        FOREIGN KEY (order_id) REFERENCES orders(uid),
        FOREIGN KEY (product_id) REFERENCES products(uid)
    )


    ### Response:
    Based on your instructions, here is the SQL query I have generated to answer the question : `{question}`:

    sql statement starts here:
    """

### send_message_llm_for_sql

In [13]:
def send_message_llm_for_sql(message, model, system_message=None, max_tokens=8000, client=ollama_client):
    """Send a message to the LLM and return the response"""
    display(Markdown(f"Model: `{model}`"))
    # display(Markdown(f"Answer to: **{message}**"))
    options = {}
    if max_tokens:
        options['num_predict'] = max_tokens
    # if streaming:
    # output = display(Markdown('Computing ...'), display_id=True)
    stream = client.chat(
        model=model,
        messages=[{'role': 'user', 'content': message}],
        stream=True,
        options=options if options else None,
        )
    streamed = ""
    for chunk in stream:
        streamed += chunk['message']['content']
        # output.update(Markdown(streamed))

    print(f"Done! in {len(streamed):,d} characters.")
    return streamed
    # else:
    # output = display(Markdown('Computing ...'), display_id=True)
    # response = ollama.chat(
    #     model=model, 
    #     messages=[{'role': 'user', 'content': message}], 
    #     stream=False,
    #     options=options if options else None,
    # )
    # output.update(Markdown(response['message']['content']))
    # print(f"Done! in {len(response['message']['content']):,d} characters.")
    # return response

### extract_sql

In [14]:
def extract_sql(response_text):
    """Extract SQL code from LLM response"""
    
    # Try to find SQL in code blocks first
    code_block_match = re.search(r'```sql\n(.*?)\n```', response_text, re.DOTALL | re.IGNORECASE)
    if code_block_match:
        return code_block_match.group(1).strip()
    
    # Try generic code blocks
    code_block_match = re.search(r'```\n(.*?)\n```', response_text, re.DOTALL)
    if code_block_match:
        return code_block_match.group(1).strip()
    
    # Look for lines starting with SQL keywords
    lines = response_text.strip().split('\n')
    sql_lines = []
    in_sql = False
    
    for line in lines:
        line_stripped = line.strip()
        # Start capturing when we see SQL keywords
        if re.match(r'^(SELECT|INSERT|UPDATE|DELETE|WITH|CREATE)', line_stripped, re.IGNORECASE):
            in_sql = True
        
        if in_sql:
            sql_lines.append(line)
            # Stop at semicolon
            if ';' in line:
                break
    
    if sql_lines:
        return '\n'.join(sql_lines).strip()
    
    # If nothing found, return the whole response
    return response_text.strip()

# Text to SQL with local LLM

### Showing the model capabilities

In [15]:
model_checkpoint = 'codellama:7b'
message = "how do i create a function in python to add a and b?"


output = send_message_to_llm(message=message, model=model_checkpoint, streaming=True)

Model: `codellama:7b`

Answer to: **how do i create a function in python to add a and b?**


To create a function in Python that adds two numbers, you can use the `def` keyword followed by the name of the function, a list of arguments (in this case, just `a` and `b`), and a colon. The function body is indented below the definition. Here's an example:
```
def add(a, b):
    return a + b
```
This function takes two arguments, `a` and `b`, adds them together using the `+` operator, and returns the result. You can call this function by passing in two numbers as arguments:
```
result = add(3, 5)
print(result) # prints 8
```
You can also use the `*` operator to unpack a tuple or list of arguments, like this:
```
def add_many(*args):
    return sum(args)

result = add_many(3, 5, 7)
print(result) # prints 15
```
This function takes any number of arguments (represented by the `*args` parameter), adds them together using the `sum()` function, and returns the result.

Done! in 878 characters.


Test the code:

In [16]:
def add(a, b):
    return a + b

print(add(3, 4))

7


In [17]:
model_checkpoint = 'codellama:7b'
message = "what is the minimum code to create a random forest classifer in scikit-learn?"


output = send_message_to_llm(message=message, model=model_checkpoint, streaming=True)

Model: `codellama:7b`

Answer to: **what is the minimum code to create a random forest classifer in scikit-learn?**


To create a random forest classifier using scikit-learn, you can use the `RandomForestClassifier` class and specify the number of trees to build. Here's an example:
```
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load the iris dataset
iris = load_iris()
X = iris.data[:, :2]  # we only take the first two features.
y = iris.target

# Split the dataset into a training set and a test set
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

# Train a random forest classifier with 100 trees
clf = RandomForestClassifier(n_estimators=100, random_state=0)
clf.fit(X_train, y_train)

# Predict the class labels for the test set
y_pred = clf.predict(X_test)
```
This code trains a random forest classifier with 100 trees on the iris dataset and then uses it to predict the class labels for the test set. You can adjust the number of trees in the forest by changing the `n_estimators` parameter.

Done! in 1,015 characters.


Test the code:

In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load the iris dataset
iris = load_iris()
X = iris.data[:, :2]  # we only take the first two features.
y = iris.target

# Split the dataset into a training set and a test set
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

# Train a random forest classifier with 100 trees
clf = RandomForestClassifier(n_estimators=100, random_state=0)
clf.fit(X_train, y_train)

# Predict the class labels for the test set
y_pred = clf.predict(X_test)

# Evaluate the model
accuracy = (y_pred == y_test).mean()
print("Accuracy:", accuracy)

Accuracy: 0.5789473684210527


### Simple query to database

In [19]:
query_db("SELECT * FROM accounts")

,uid,account_name,city,country
0,1,TechStyle Inc,Tokyo,Japan
1,2,FashionHub Inc,Shenzhen,PRC
2,3,ModernWear Inc,Shenzhen,PRC
3,4,GlobalTextile Inc,Shanghai,PRC
4,5,UrbanThread Inc,Shanghai,PRC
5,6,StyleCraft Inc,Shanghai,PRC
6,7,PremiumFabric Inc,Tokyo,Japan
7,8,ElegantWear Inc,Beijing,PRC
8,9,TrendSetter Inc,Beijing,PRC
9,10,ChicOutfit Inc,Tokyo,Japan


In [20]:
model_checkpoint = 'codellama:7b'
message = f"""
This is a database scheme: {db_schema}. 
Give me the sql statement to list all companies in the database.
"""

output = send_message_llm_for_sql(message=message, model=model_checkpoint)
print('\nLLM output:')
display(Markdown(f"```ascii\n{output}```"))
print('')
print('Extracted Code:')
display(Markdown(f"```sql\n{extract_sql(output)}\n```"))

Model: `codellama:7b`

Done! in 365 characters.

LLM output:


```ascii
Here is the SQL statement to list all companies in the database:
```
SELECT * FROM accounts;
```
This will return all rows from the "accounts" table, which represents the companies in your database. The `*` wildcard selects all columns in the table, so you will get a list of all companies and their associated information (such as account name, city, and country).```


Extracted Code:


```sql
SELECT * FROM accounts;
```

Test the SQL:

In [21]:
sql = extract_sql(output)

query_db(sql)

,uid,account_name,city,country
0,1,TechStyle Inc,Tokyo,Japan
1,2,FashionHub Inc,Shenzhen,PRC
2,3,ModernWear Inc,Shenzhen,PRC
3,4,GlobalTextile Inc,Shanghai,PRC
4,5,UrbanThread Inc,Shanghai,PRC
5,6,StyleCraft Inc,Shanghai,PRC
6,7,PremiumFabric Inc,Tokyo,Japan
7,8,ElegantWear Inc,Beijing,PRC
8,9,TrendSetter Inc,Beijing,PRC
9,10,ChicOutfit Inc,Tokyo,Japan


In [22]:
model_checkpoint = 'codellama:7b'
co_name = 'FashionHub Inc'

message = f"Give me the sql statement to get all orders which were placed by {co_name} with all details."


output = send_message_llm_for_sql(message=message, model=model_checkpoint)
print('\nLLM output:')
display(Markdown(f"```ascii\n{output}```"))
print('')
print('Extracted Code:')
display(Markdown(f"```sql\n{extract_sql(output)}\n```"))

Model: `codellama:7b`

Done! in 752 characters.

LLM output:


```ascii

To get all orders placed by FashionHub Inc with all details, you can use a SQL query like this:
```
SELECT * 
FROM Orders
WHERE CustomerName = 'FashionHub Inc';
```
This will retrieve all orders that were placed by FashionHub Inc and include all of the order details in the result set.

It's important to note that you should replace `'FashionHub Inc'` with the actual name of the customer who placed the orders.

Also, if you want to retrieve specific columns only, you can use a SELECT clause like this:
```
SELECT OrderID, CustomerName, TotalPrice, OrderDate 
FROM Orders
WHERE CustomerName = 'FashionHub Inc';
```
This will retrieve only the `OrderID`, `CustomerName`, `TotalPrice`, and `OrderDate` columns for all orders placed by FashionHub Inc.```


Extracted Code:


```sql
SELECT * 
FROM Orders
WHERE CustomerName = 'FashionHub Inc';
```

In [23]:
try:
    result = query_db(extract_sql(output))
    display(result)
except Exception as e:
    print(f"Error executing SQL: \n{e}")

Error executing SQL: 
Execution failed on sql 'SELECT * 
FROM Orders
WHERE CustomerName = 'FashionHub Inc';': no such column: CustomerName


In [24]:
# Query all order details for FashionHub Inc
co_name = 'FashionHub Inc'
sql = f"""
SELECT 
    a.account_name,
    a.city,
    a.country,
    o.uid as order_id,
    o.order_date,
    o.status,
    p.product_name,
    p.product_price,
    od.quantity,
    (p.product_price * od.quantity) as line_total
FROM accounts a
JOIN orders o ON a.uid = o.account_id
JOIN order_details od ON o.uid = od.order_id
JOIN products p ON od.product_id = p.uid
WHERE a.account_name = "{co_name}"
ORDER BY o.order_date DESC, o.uid
"""
df = query_db(sql)
print(f"{co_name} ordered a total of {df.line_total.sum():,.2f}")
display(df)

FashionHub Inc ordered a total of 2,299.64


,account_name,city,country,order_id,order_date,status,product_name,product_price,quantity,line_total
0,FashionHub Inc,Shenzhen,PRC,1,2024-09-04,Delivered,Pencil Skirt,44.99,10,449.90
1,FashionHub Inc,Shenzhen,PRC,46,2024-04-18,Delivered,Hooded Sweatshirt,54.99,6,329.94
2,FashionHub Inc,Shenzhen,PRC,46,2024-04-18,Delivered,Wool Sweater,79.99,9,719.91
3,FashionHub Inc,Shenzhen,PRC,46,2024-04-18,Delivered,Cotton T-Shirt,19.99,4,79.96
4,FashionHub Inc,Shenzhen,PRC,46,2024-04-18,Delivered,Leather Jacket,149.99,4,599.96
5,FashionHub Inc,Shenzhen,PRC,46,2024-04-18,Delivered,Track Pants,39.99,3,119.97


### Agentic Version of SQL LLM

In [25]:
def llm_query_loop(message, max_attempts=10, model='codellama:7b', client=ollama_client):
    """Loop to refine SQL query based on LLM feedback"""
    attempt = 0
    correct_sql = False
    outputs = []
    sqls = []
    while not correct_sql and attempt < max_attempts:
        display(Markdown(f"**Attempt {attempt+1}**:"))
        # Get LLM response
        output = send_message_llm_for_sql(message=message, model=model)   
        outputs.append(output)
        # Extract SQL
        sql = extract_sql(output)
        sqls.append(sql)
        display(Markdown(f"```sql\n{sql}\n```"))
        # Test it
        try:
            result = query_db(sql)
            print(f"\n✓ Query successful!")
            correct_sql = True
            return sql, result, sqls, outputs
        except Exception as e:
            print(f"\n✗ Query {attempt+1} failed test!")
            attempt += 1
            # Modify the user message to provide the question, the wrong SQL and the error message
            message = f"""
            This is a database schema: {db_schema}
            Generate a SQL statement that answer the following question:
                Question: {question}
            Return ONLY the SQL statement without explanation. Start directly with SELECT/INSERT/UPDATE/DELETE.
            
            Generated SQL statement:
            ```sql
            {sql}
            ```
            When running the above SQL, the following error was raised: {e}

            Correct the SQL statement and provide the corrected SQL statement for the question.
            Return ONLY the SQL statement without explanation. Start directly with SELECT/INSERT/UPDATE/DELETE.
            """
    print("\n✗ Query failed for too many attempts.")
    return None, None, sqls, outputs


In [26]:
def ask_llm_for_sql(question, model='codellama:7b'):
    """Ask LLM for SQL and automatically extract and test it"""
    message = f"""
    This is a database schema: {db_schema}
    Generate a SQL statement that answer the following question:
        Question: {question}
    Return ONLY the SQL statement without explanation. Start directly with SELECT/INSERT/UPDATE/DELETE.
    """

    max_attempts = 10
    sql, result, sqls, outputs = llm_query_loop(message, max_attempts, model)
    if sql:
        return sql, result, sqls, outputs
    else:
        sql, result, sqls, outputs = llm_query_loop(message, max_attempts, model)
        if sql:
            return sql, result, sqls, outputs
        else:
            print(f"\n✗ Query {max_attempts} attempts failed test!")    
            return None, None, sqls, outputs

In [27]:
question = "List of all the companies"

sql, result, sqls, output = ask_llm_for_sql(question)

if result is not None:
    display(result)

**Attempt 1**:

Model: `codellama:7b`

Done! in 35 characters.


```sql
SELECT account_name FROM accounts;
```


✓ Query successful!


,account_name
0,TechStyle Inc
1,FashionHub Inc
2,ModernWear Inc
3,GlobalTextile Inc
4,UrbanThread Inc
5,StyleCraft Inc
6,PremiumFabric Inc
7,ElegantWear Inc
8,TrendSetter Inc
9,ChicOutfit Inc


In [30]:
model_checkpoint = 'codellama:7b'
co_name = 'FashionHub Inc'

question = f"Give me the sql statement to get all orders which were placed by {co_name} with all details."

sql, result, sqls, output = ask_llm_for_sql(question)

if result is not None:
    display(result)

**Attempt 1**:

Model: `codellama:7b`

Done! in 279 characters.


```sql
SELECT
    o.uid,
    o.account_id,
    o.order_date,
    o.status,
    od.product_name,
    od.product_price,
    od.quantity
FROM orders AS o
JOIN order_details AS od ON o.uid = od.order_id
WHERE o.account_id IN (SELECT uid FROM accounts WHERE account_name = 'FashionHub Inc')
```


✗ Query 1 failed test!


**Attempt 2**:

... (6 outputs left out)

Done! in 516 characters.


```sql
SELECT o.uid,
       o.account_id,
       o.order_date,
       o.status,
       p.product_name,
       p.product_price,
       od.quantity
FROM orders AS o
JOIN order_details AS od ON o.uid = od.order_id
JOIN products AS p ON od.product_id = p.uid
WHERE o.account_id IN (SELECT uid FROM accounts WHERE account_name = 'FashionHub Inc')
```


✓ Query successful!


,uid,account_id,order_date,status,product_name,product_price,quantity
0,1,2,2024-09-04,Delivered,Pencil Skirt,44.99,10
1,46,2,2024-04-18,Delivered,Hooded Sweatshirt,54.99,6
2,46,2,2024-04-18,Delivered,Wool Sweater,79.99,9
3,46,2,2024-04-18,Delivered,Cotton T-Shirt,19.99,4
4,46,2,2024-04-18,Delivered,Leather Jacket,149.99,4
5,46,2,2024-04-18,Delivered,Track Pants,39.99,3


In [34]:
question = """
Get the list of all orders with all order and product details for account FashionHub Inc. 
Add the total price per order line. 
Order them by ascending order date
"""

sql, result, sqls, outputs = ask_llm_for_sql(question)

if result is not None:
    display(result)

**Attempt 1**:

Model: `codellama:7b`

Done! in 645 characters.


```sql
SELECT orders.uid,
         orders.order_date,
         orders.status,
         order_details.uid AS detail_uid,
         order_details.order_id,
         order_details.product_id,
         order_details.quantity,
         products.product_name,
         products.product_price,
         (products.product_price * order_details.quantity) AS total_price
   FROM orders
     INNER JOIN order_details ON orders.uid = order_details.order_id
     INNER JOIN products ON order_details.product_id = products.uid
     WHERE orders.account_id IN (SELECT uid FROM accounts WHERE account_name LIKE '%FashionHub Inc%')
ORDER BY orders.order_date ASC;
```


✓ Query successful!


,uid,order_date,status,detail_uid,order_id,product_id,quantity,product_name,product_price,total_price
0,5,2024-05-23,Delivered,9,5,11,9,Blazer,129.99,1169.91
1,5,2024-05-23,Delivered,10,5,12,7,Pencil Skirt,44.99,314.93
2,5,2024-05-23,Delivered,11,5,5,3,Silk Blouse,89.99,269.97
3,46,2024-06-29,Cancelled,135,46,5,1,Silk Blouse,89.99,89.99
4,46,2024-06-29,Cancelled,136,46,15,9,Chinos,54.99,494.91
5,46,2024-06-29,Cancelled,137,46,19,10,Trench Coat,179.99,1799.90
6,46,2024-06-29,Cancelled,138,46,5,6,Silk Blouse,89.99,539.94


In [35]:
# Query all order details for FashionHub Inc
co_name = 'FashionHub Inc'
sql = f"""
SELECT 
    a.account_name,
    a.city,
    a.country,
    o.uid as order_id,
    o.order_date,
    o.status,
    p.product_name,
    p.product_price,
    od.quantity,
    (p.product_price * od.quantity) as line_total
FROM accounts a
JOIN orders o ON a.uid = o.account_id
JOIN order_details od ON o.uid = od.order_id
JOIN products p ON od.product_id = p.uid
WHERE a.account_name = "{co_name}"
ORDER BY o.order_date ASC
"""
df = query_db(sql)
print(f"{co_name} ordered a total of {df.line_total.sum():,.2f}")
display(df)

FashionHub Inc ordered a total of 4,679.55


,account_name,city,country,order_id,order_date,status,product_name,product_price,quantity,line_total
0,FashionHub Inc,Shenzhen,PRC,5,2024-05-23,Delivered,Blazer,129.99,9,1169.91
1,FashionHub Inc,Shenzhen,PRC,5,2024-05-23,Delivered,Pencil Skirt,44.99,7,314.93
2,FashionHub Inc,Shenzhen,PRC,5,2024-05-23,Delivered,Silk Blouse,89.99,3,269.97
3,FashionHub Inc,Shenzhen,PRC,46,2024-06-29,Cancelled,Silk Blouse,89.99,1,89.99
4,FashionHub Inc,Shenzhen,PRC,46,2024-06-29,Cancelled,Chinos,54.99,9,494.91
5,FashionHub Inc,Shenzhen,PRC,46,2024-06-29,Cancelled,Trench Coat,179.99,10,1799.90
6,FashionHub Inc,Shenzhen,PRC,46,2024-06-29,Cancelled,Silk Blouse,89.99,6,539.94


In [36]:
question = """
List all orders for account FashionHub Inc, including the total price for each order. 
List the following fields: account name, order date, order ID and total price of the order
"""

sql, result, sqls, outputs = ask_llm_for_sql(question)

if result is not None:
    display(result)

**Attempt 1**:

Model: `codellama:7b`

Done! in 490 characters.


```sql
SELECT DISTINCT a.account_name AS "Account Name", o.order_date AS "Order Date", o.uid AS "Order ID", SUM(od.quantity * p.product_price) AS "Total Price" 
        FROM accounts AS a 
            INNER JOIN orders AS o ON o.account_id = a.uid 
                INNER JOIN order_details AS od ON od.order_id = o.uid 
                    INNER JOIN products AS p ON p.uid = od.product_id 
        WHERE a.account_name = 'FashionHub Inc' 
            GROUP BY o.uid ORDER BY "Order Date" ASC;
```


✓ Query successful!


,Account Name,Order Date,Order ID,Total Price
0,FashionHub Inc,2024-05-23,5,1754.81
1,FashionHub Inc,2024-06-29,46,2924.74


In [37]:
question = """
List all orders with all order details for accounts FashionHub Inc and for PremiumFabric Inc.
Show the following fields: account name, order date, order ID, product name, product price, quantity and line total.
Order by account ID first, then by order date.
"""

sql, result, sqls, outputs = ask_llm_for_sql(question)

if result is not None:
    display(result)

**Attempt 1**:

Model: `codellama:7b`

Done! in 473 characters.


```sql
SELECT orders.account_id, orders.order_date, orders.uid AS order_id, products.product_name, products.product_price, order_details.quantity, order_details.line_total
FROM orders
JOIN order_details ON orders.uid = order_details.order_id
JOIN products ON order_details.product_id = products.uid
WHERE orders.account_id IN (SELECT uid FROM accounts WHERE account_name LIKE 'FashionHub Inc' OR account_name LIKE 'PremiumFabric Inc')
ORDER BY orders.account_id, orders.order_date
```


✗ Query 1 failed test!


**Attempt 2**:

... (6 outputs left out)

Done! in 803 characters.


```sql
SELECT orders.account_id, orders.order_date, orders.uid AS order_id, products.product_name, products.product_price, order_details.quantity, (products.product_price * order_details.quantity) AS line_total
FROM orders
JOIN order_details ON orders.uid = order_details.order_id
JOIN products ON order_details.product_id = products.uid
WHERE orders.account_id IN (SELECT uid FROM accounts WHERE account_name LIKE 'FashionHub Inc' AND account_name LIKE 'PremiumFabric Inc')
ORDER BY orders.account_id, orders.order_date;
```


✓ Query successful!


,account_id,order_date,order_id,product_name,product_price,quantity,line_total


# End